In [ ]:
import os
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"]="false"
os.environ["XLA_PYTHON_CLIENT_ALLOCATOR"]="platform"
os.environ["JAX_ENABLE_X64"]="false"

import multiprocessing
os.environ["XLA_FLAGS"]="--xla_cpu_multi_thread_eigen=true intra_op_parallelism_threads=16 --xla_force_host_platform_device_count={}".format(multiprocessing.cpu_count())
# os.environ["XLA_FLAGS"]="--xla_cpu_multi_thread_eigen=true intra_op_parallelism_threads=16"


from sampling import *
from wavefunction import *
from observables import *
from training import *
from utils import *
import time

from plotly import graph_objects as go



fig = go.FigureWidget()

# Energy mean
fig.add_trace(
    go.Scatter(
        x=[], y=[],
        mode='lines+markers',
        name='Energy',
        line=dict(color='blue')
    )
)

# Upper uncertainty bound: E + σ
fig.add_trace(
    go.Scatter(
        x=[], y=[],
        mode='lines',
        line=dict(width=0),
        name='Energy + σ',
        showlegend=False
    )
)

# Lower uncertainty bound: E - σ (filled to previous trace)
fig.add_trace(
    go.Scatter(
        x=[], y=[],
        mode='lines',
        fill='tonexty',
        fillcolor='rgba(0, 0, 255, 0.2)',
        line=dict(width=0),
        name='Energy ± σ'
    )
)

fig.update_layout(
    title='Energy with Uncertainty Band over Training Steps',
    xaxis_title='Training Step',
    yaxis_title='Energy'
)

In [ ]:
N = 5
eta = 1
g = 1
N_CORES = 16

x = np.random.normal(size=(N, 3))
model = MLP_TI(hidden_sizes=(64,64))
# model = TI_CNN()
params = model.init(jax.random.PRNGKey(int(time.time())), x)
y = model.apply(params,x)

@jit
def psi(params, config):
    return jnp.exp(-model.apply(params, config))

In [ ]:
sampler = Sampler(psi, (N, 3))

nchains = N_CORES
pos_initials = [jnp.zeros((N, 3)) for _ in range(nchains)]
seeds = jnp.arange(nchains)
var = .5 # Modify this

samples, acc_rate = sampler.run_many_chains(params, nchains**4//nchains, 1000, 3, var, pos_initials, seeds)
print(f"Acceptance rate: {acc_rate}")
print(samples.shape)

Acceptance rate: 0.5744562745094299
(65536, 5, 3)


In [11]:
# show fig
fig.show()

FigureWidget({
    'data': [{'line': {'color': 'blue'},
              'mode': 'lines+markers',
              'name': 'Energy',
              'type': 'scatter',
              'uid': '47a0059f-1efe-462c-b7fd-58dc1dc3da27',
              'x': [],
              'y': []},
             {'line': {'width': 0},
              'mode': 'lines',
              'name': 'Energy + σ',
              'showlegend': False,
              'type': 'scatter',
              'uid': 'da0b3645-3bae-4d91-9675-13e88021e4a7',
              'x': [],
              'y': []},
             {'fill': 'tonexty',
              'fillcolor': 'rgba(0, 0, 255, 0.2)',
              'line': {'width': 0},
              'mode': 'lines',
              'name': 'Energy ± σ',
              'type': 'scatter',
              'uid': 'c8a8e2c7-9d6e-42fc-8da7-57e5f580f071',
              'x': [],
              'y': []}],
    'layout': {'template': '...',
               'title': {'text': 'Energy with Uncertainty Band over Training Steps'},
    

In [ ]:
init_params = params
lr = 1e-3
MC_options= {
    "num_samples": 16**3,
    "thermalization": 1000,
    "skip": 2,
    "var": var,
    "nchains": 16,
    "seeds": [int(time.time()) + i for i in range(16)], # current time plus the index of the chain
    "pos_initials": [jnp.zeros((N, 3)) for _ in range(nchains)],
}
steps = 10000


resultsa = train(init_params, model, eta, g, sampler, MC_options, steps, lr, fig=fig)

Energy = -0.39393752813339233:   4%|▍         | 395/10000 [01:04<29:04,  5.51it/s]

In [ ]:
# save the parameters to a file
np.save("params.npy", resultsa[0])
# load them
params = np.load("params.npy", allow_pickle=True).item()